In [6]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import jax
import jax.numpy as jnp
from jax import jit, jacobian
jax.config.update("jax_enable_x64", True)
import dolfinx
from dolfinx.mesh import create_interval, locate_entities_boundary
from dolfinx.fem import (
    functionspace, Function,
    locate_dofs_topological,)
from mpi4py import MPI

In [16]:
MU      = 100e3   # elastic shear modulus [MPa]
S0      = 100.0   # initial / coarse-grain yield strength [MPa]
D0      = 0.1     # reference flow rate [s^-1]
M_RATE  = 0.02    # rate-sensitivity exponent
H_STRIP = 1.0     # strip height (normalised)

GAMMA_MAX  = 0.10
GAMMA_RATE = 0.01
T_END_FWD  = GAMMA_MAX / GAMMA_RATE   # 10 s

N_STEPS_FWD = 50
N_STEPS_REV = 50

N_ELEM    = 20       # quadratic elements  => 2*N_ELEM+1 nodes
N_NODES_NP = 2*N_ELEM + 1
N_DOF_NP   = 2 * N_NODES_NP
EPS_DP    = 1e-12    # regularise sqrt denominator

NR_MAX_ITER = 50
NR_TOL      = 1e-8

# 3-pt Gauss rule on [-1,+1]
XI_GP = np.array([-np.sqrt(3.0 / 5.0), 0.0, np.sqrt(3.0 / 5.0)])
W_GP  = np.array([5.0 / 9.0,           8.0 / 9.0, 5.0 / 9.0])

In [17]:
@jit
def _dp_jax(gp_dot: jnp.float64,
             gp_dot_y: jnp.float64,
             l_dis: jnp.float64) -> jnp.float64:
    """Regularised effective plastic flow rate (scalar)."""
    return jnp.sqrt(gp_dot**2 + l_dis**2 * gp_dot_y**2 + EPS_DP**2)

In [18]:
@jit
def constitutive_jax(gp_dot, gp_dot_y, S, l_dis, L_en):
    """
    Microstresses at a single Gauss point (all JAX scalars).

    Returns
    -------
    tau_p  : scalar microstress  [MPa]
    kdis   : dissipative gradient microstress  [MPa·mm]
    ken    : energetic gradient microstress     [MPa·mm]
    dp     : effective flow rate  [s^-1]
    """
    dp      = _dp_jax(gp_dot, gp_dot_y, l_dis)
    ratio_m = (dp / D0) ** M_RATE
    tau_p   = S  * ratio_m * (gp_dot   / dp)
    kdis    = S0 * l_dis**2 * ratio_m * (gp_dot_y / dp)
    ken     = S0 * L_en**2  * gp_dot_y           # energetic: rate-independent
    return tau_p, kdis, ken, dp

In [19]:
def _micro_vec(args, S, l_dis, L_en):
    """Helper returning [tau_p, kdis] as a (2,) array for jax.jacobian."""
    tau_p, kdis, ken, dp = constitutive_jax(args[0], args[1], S, l_dis, L_en)
    return jnp.stack([tau_p, kdis])

In [20]:
@jit
def constitutive_tangent_jax(gp_dot, gp_dot_y, S, l_dis, L_en):
    """
    Exact tangent  J[i,j] = d [tau_p, kdis][i] / d [gp_dot, gp_dot_y][j]
    Shape: (2, 2). Computed via JAX autodiff (no hand differentiation).
    """
    return jacobian(lambda a: _micro_vec(a, S, l_dis, L_en))(
        jnp.array([gp_dot, gp_dot_y]))

In [21]:
@jit
def update_S_jax(S_old, dp, dt, H_hard):
    """Backward-Euler isotropic hardening:  S_new = S_old + H * dp * dt."""
    return S_old + H_hard * dp * dt

In [22]:
def shape_functions(xi: float):
    N  = np.array([0.5*xi*(xi-1.), 1.-xi**2,  0.5*xi*(xi+1.)])
    dN = np.array([xi - 0.5,      -2.*xi,     xi + 0.5])
    return N, dN

In [23]:
def build_mesh_np(n_elem: int, h_strip: float):
    nodes = np.linspace(0., h_strip, 2*n_elem + 1)
    conn  = np.array([[2*e, 2*e+1, 2*e+2] for e in range(n_elem)], dtype=int)
    ldof  = np.array([[2*cn[a]+k for a in range(3) for k in range(2)]
                      for cn in conn], dtype=int)
    return nodes, conn, ldof

In [24]:
def element_res_jac(u_e, gp_e, gp_pe, S_gp_e, y_e,
                    l_dis, L_en, tau_glob, dt):
    
    h_e = y_e[-1] - y_e[0]
    jac = h_e / 2.0

    r_e = np.zeros(6)
    K_e = np.zeros((6, 6))

    for igp, (xi, w) in enumerate(zip(XI_GP, W_GP)):
        N, dN_xi = shape_functions(xi)
        dN = dN_xi / jac   # dN/dy

        gp_val    = float(N  @ gp_e)
        gp_prev_v = float(N  @ gp_pe)
        gp_y      = float(dN @ gp_e)
        gp_prev_y = float(dN @ gp_pe)

        gp_dot   = (gp_val   - gp_prev_v) / dt
        gp_dot_y = (gp_y     - gp_prev_y) / dt

        S = float(S_gp_e[igp])

        # ── JAX constitutive (JIT-compiled) ──────────────────────────────────
        tau_p, kdis, ken, dp = constitutive_jax(
            jnp.float64(gp_dot), jnp.float64(gp_dot_y),
            jnp.float64(S), jnp.float64(l_dis), jnp.float64(L_en))
        tau_p = float(tau_p); kdis = float(kdis)
        ken   = float(ken);   dp   = float(dp)
        kp    = ken + kdis

        # ── JAX tangent (autodiff, exact) ─────────────────────────────────────
        # J2[0,:] = d(tau_p)/d([gp_dot, gp_dot_y])
        # J2[1,:] = d(kdis) /d([gp_dot, gp_dot_y])
        J2 = np.array(constitutive_tangent_jax(
            jnp.float64(gp_dot), jnp.float64(gp_dot_y),
            jnp.float64(S), jnp.float64(l_dis), jnp.float64(L_en)))

        # chain rule to nodal DOFs (backward-Euler: d(gp_dot)/d(gp_B) = N_B/dt)
        dtaup_dgpB = J2[0, 0] * N / dt + J2[0, 1] * dN / dt   # (3,)
        dkdis_dgpB = J2[1, 0] * N / dt + J2[1, 1] * dN / dt   # (3,)
        dken_dgpB  = S0 * L_en**2 * dN                          # (3,)
        dkp_dgpB   = dkdis_dgpB + dken_dgpB                     # (3,)

        vol = w * jac

        # ── Residuals ─────────────────────────────────────────────────────────
        r_e[0::2] += vol * tau_glob * dN                        # macro (u-DOFs)
        r_e[1::2] += vol * ((tau_p - tau_glob) * N + kp * dN)  # micro (gp-DOFs)

        # ── Jacobian ──────────────────────────────────────────────────────────
        for A in range(3):
            for B in range(3):
                # K_uu:  d(R_u_A)/d(u_B)
                K_e[2*A,   2*B  ] += vol * MU * dN[A] * dN[B]
                # K_u_gp: d(R_u_A)/d(gp_B) = -mu * N_A,y * N_B
                K_e[2*A,   2*B+1] -= vol * MU * dN[A] * N[B]
                # K_gp_u: d(R_gp_A)/d(u_B) = -mu * N_A * N_B,y
                K_e[2*A+1, 2*B  ] -= vol * MU * N[A]  * dN[B]
                # K_gp_gp: d(R_gp_A)/d(gp_B)  (JAX autodiff tangent)
                K_e[2*A+1, 2*B+1] += vol * (dtaup_dgpB[B] * N[A]
                                            + dkp_dgpB[B]  * dN[A])
    return r_e, K_e

In [25]:
def assemble_np(nodes, conn, ldof, u_vec, gp_vec, gp_prev, S_all,
                l_dis, L_en, tau_glob, dt):
    R = np.zeros(N_DOF_NP)
    K = np.zeros((N_DOF_NP, N_DOF_NP))
    for e, (cn, dofs) in enumerate(zip(conn, ldof)):
        r_e, K_e = element_res_jac(
            u_vec[cn], gp_vec[cn], gp_prev[cn],
            S_all[e], nodes[cn], l_dis, L_en, tau_glob, dt)
        R[dofs] += r_e
        np.add.at(K, (dofs[:, None], dofs[None, :]), K_e)
    return R, K

In [26]:
def apply_bcs_np(K, R, fixed_dofs):
    Km = K.copy(); Rm = R.copy()
    for fd in fixed_dofs:
        Km[fd, :] = 0.; Km[:, fd] = 0.
        Km[fd, fd] = 1.; Rm[fd]   = 0.
    return Km, Rm

In [27]:
def compute_gamma_p_np(nodes, conn, gp_vec):
    """Average plastic shear strain Gamma_p = (1/h)*integral gp dy."""
    Gp = 0.
    for e, cn in enumerate(conn):
        y_e  = nodes[cn]; gp_e = gp_vec[cn]
        jac  = (y_e[-1] - y_e[0]) / 2.
        for xi, w in zip(XI_GP, W_GP):
            N, _ = shape_functions(xi)
            Gp  += w * jac * float(N @ gp_e)
    return Gp / H_STRIP

In [28]:
def update_gp_state_np(nodes, conn, gp_new, gp_prev, S_all, l_dis, H_hard, dt):
    """Update flow resistance S at each Gauss point (backward-Euler)."""
    S_new = S_all.copy()
    for e, cn in enumerate(conn):
        y_e = nodes[cn]; jac = (y_e[-1] - y_e[0]) / 2.
        for igp, xi in enumerate(XI_GP):
            N, dN_xi = shape_functions(xi); dN = dN_xi / jac
            gp_dot   = float((N  @ gp_new[cn] - N  @ gp_prev[cn]) / dt)
            gp_dot_y = float((dN @ gp_new[cn] - dN @ gp_prev[cn]) / dt)
            dp = float(_dp_jax(jnp.float64(gp_dot), jnp.float64(gp_dot_y),
                                jnp.float64(l_dis)))
            S_new[e, igp] = float(update_S_jax(
                jnp.float64(S_all[e, igp]), jnp.float64(dp),
                jnp.float64(dt), jnp.float64(H_hard)))
    return S_new

In [29]:
# NEWTON-RAPHSON STEP (NumPy path)
# ═════════════════════════════════════════════════════════════════════════════

def solve_step_np(nodes, conn, ldof, u_vec, gp_vec, gp_prev, S_all,
                  l_dis, L_en, H_hard, Gamma_imposed, dt):
    u_new  = u_vec.copy()
    gp_new = gp_vec.copy()

    # Boundary DOFs
    dof_u0   = 0
    dof_uh   = 2 * (N_NODES_NP - 1)
    dof_gp0  = 1
    dof_gph  = 2 * (N_NODES_NP - 1) + 1
    fixed    = [dof_u0, dof_uh, dof_gp0, dof_gph]

    for _ in range(NR_MAX_ITER):
        Gp  = compute_gamma_p_np(nodes, conn, gp_new)
        tau = MU * (Gamma_imposed - Gp)

        R, K = assemble_np(nodes, conn, ldof, u_new, gp_new, gp_prev,
                            S_all, l_dis, L_en, tau, dt)
        Km, Rm = apply_bcs_np(K, R, fixed)

        if np.linalg.norm(Rm) < NR_TOL:
            break

        delta  = np.linalg.solve(Km, -Rm)
        u_new  += delta[0::2]
        gp_new += delta[1::2]

        # Re-impose Dirichlet
        u_new[0]              = 0.
        u_new[N_NODES_NP - 1] = Gamma_imposed * H_STRIP
        gp_new[0]             = gp_prev[0]
        gp_new[N_NODES_NP-1]  = gp_prev[N_NODES_NP-1]

    S_new = update_gp_state_np(nodes, conn, gp_new, gp_prev, S_all,
                                l_dis, H_hard, dt)
    return u_new, gp_new, S_new, tau

In [30]:
# TIME INTEGRATION DRIVER (NumPy path)
# ═════════════════════════════════════════════════════════════════════════════

def run_simulation_np(l_dis, L_en, H_hard,
                      n_steps_fwd=N_STEPS_FWD, n_steps_rev=N_STEPS_REV,
                      reverse=True):
    nodes, conn, ldof = build_mesh_np(N_ELEM, H_STRIP)
    u_vec  = np.zeros(N_NODES_NP)
    gp_vec = np.zeros(N_NODES_NP)
    S_all  = np.full((N_ELEM, 3), S0)

    Gamma_hist = [0.]; tau_hist = [0.]; gp_profile = None
    dt_fwd = T_END_FWD / n_steps_fwd
    dt_rev = T_END_FWD / n_steps_rev

    for step in range(1, n_steps_fwd + 1):
        Gamma = GAMMA_MAX * step / n_steps_fwd
        u_vec, gp_vec, S_all, tau = solve_step_np(
            nodes, conn, ldof, u_vec, gp_vec, gp_vec.copy(),
            S_all, l_dis, L_en, H_hard, Gamma, dt_fwd)
        Gamma_hist.append(Gamma); tau_hist.append(tau)
        if step == n_steps_fwd:
            gp_profile = (nodes.copy(), gp_vec.copy())

    if reverse:
        for step in range(1, n_steps_rev + 1):
            Gamma = GAMMA_MAX - 2.*GAMMA_MAX * step / n_steps_rev
            gp_prev = gp_vec.copy()
            u_vec, gp_vec, S_all, tau = solve_step_np(
                nodes, conn, ldof, u_vec, gp_vec, gp_prev,
                S_all, l_dis, L_en, H_hard, Gamma, dt_rev)
            Gamma_hist.append(Gamma); tau_hist.append(tau)

    return np.array(Gamma_hist), np.array(tau_hist), gp_profile

In [34]:
def run_simulation_fenicsx(l_dis, L_en, H_hard,
                            n_steps_fwd=N_STEPS_FWD, n_steps_rev=N_STEPS_REV,
                            reverse=True):
    """
    DOLFINx-based simulation.

    Mesh / DOF infrastructure : DOLFINx (CG2 on interval)
    Constitutive kernel        : JAX (JIT + autodiff tangent)
    Assembly                   : Python loop (same as NumPy path, uses FEniCSx DOF coords)
    Linear solve per NR step   : numpy.linalg.solve (swap to petsc4py KSP for large runs)

    TODO: wrap constitutive_jax as dolfinx ExternalOperator for full UFL integration.
    """
    comm = MPI.COMM_WORLD

    # ── DOLFINx mesh (CG2 on [0, H_STRIP], N_ELEM cells) ─────────────────────
    msh = create_interval(comm, N_ELEM, np.array([0.0, H_STRIP]))

    V_u  = functionspace(msh, ("Lagrange", 2))
    V_gp = functionspace(msh, ("Lagrange", 2))

    # Extract DOF coordinate arrays (sorted along y)
    x_u_dof  = V_u.tabulate_dof_coordinates()[:, 0]
    x_gp_dof = V_gp.tabulate_dof_coordinates()[:, 0]

    sort_u  = np.argsort(x_u_dof)
    sort_gp = np.argsort(x_gp_dof)

    x_u_sorted  = x_u_dof[sort_u]    # (2*N_ELEM+1,)
    x_gp_sorted = x_gp_dof[sort_gp]

    n_u  = len(x_u_sorted)
    n_gp = len(x_gp_sorted)

    # ── Element connectivity in sorted DOF space ───────────────────────────────
    # CG2 on N_ELEM cells: sorted DOFs are [0, mid0, 1, mid1, ..., N_ELEM]
    # Triples: (2e, 2e+1, 2e+2) in sorted order
    conn_u  = np.array([[2*e, 2*e+1, 2*e+2] for e in range(N_ELEM)], dtype=int)
    conn_gp = np.array([[2*e, 2*e+1, 2*e+2] for e in range(N_ELEM)], dtype=int)
    # (u and gp live on the same mesh, so connectivity is identical)

    # ── Boundary DOFs (in sorted arrays) ──────────────────────────────────────
    def boundary_dofs_sorted(x_sorted, val, tol=1e-12):
        return np.where(np.abs(x_sorted - val) < tol)[0]

    dofs_u_L   = boundary_dofs_sorted(x_u_sorted,  0.)
    dofs_u_R   = boundary_dofs_sorted(x_u_sorted,  H_STRIP)
    dofs_gp_L  = boundary_dofs_sorted(x_gp_sorted, 0.)
    dofs_gp_R  = boundary_dofs_sorted(x_gp_sorted, H_STRIP)

    # ── State arrays ──────────────────────────────────────────────────────────
    u_arr   = np.zeros(n_u)
    gp_arr  = np.zeros(n_gp)
    S_all   = np.full((N_ELEM, 3), S0)

    # ── Helper: average plastic strain ────────────────────────────────────────
    def compute_gamma_p_fnx(gp_a):
        Gp = 0.
        for e in range(N_ELEM):
            y_e   = x_u_sorted[conn_u[e]]   # coords from u mesh (same as gp)
            gp_e  = gp_a[conn_gp[e]]
            jac_e = (y_e[-1] - y_e[0]) / 2.
            for xi, w in zip(XI_GP, W_GP):
                N, _ = shape_functions(xi)
                Gp  += w * jac_e * float(N @ gp_e)
        return Gp / H_STRIP

    # ── Helper: element residual / Jacobian (reuses element_res_jac) ──────────
    def assemble_fnx(u_a, gp_a, gp_prev_a, S_in, tau_glob, dt):
        """
        Returns R (n_u+n_gp,) and K ((n_u+n_gp)^2) in block layout [u|gp].
        """
        R_u  = np.zeros(n_u)
        R_gp = np.zeros(n_gp)
        K_uu  = np.zeros((n_u,  n_u))
        K_ugp = np.zeros((n_u,  n_gp))
        K_gpu = np.zeros((n_gp, n_u))
        K_gp  = np.zeros((n_gp, n_gp))

        for e in range(N_ELEM):
            cn_u  = conn_u[e]
            cn_gp = conn_gp[e]
            y_e   = x_u_sorted[cn_u]

            # Pack element arrays (same layout as numpy path's ldof [u0,gp0,...])
            # We call element_res_jac which expects interleaved u/gp layout.
            # Rebuild interleaved element arrays:
            u_e   = u_a[cn_u]
            gp_e  = gp_a[cn_gp]
            gppe  = gp_prev_a[cn_gp]

            # Build 6-entry interleaved DOF arrays for element_res_jac
            u_e_6   = np.zeros(3); gp_e_6 = np.zeros(3); gppe_6 = np.zeros(3)
            u_e_6[:] = u_e; gp_e_6[:] = gp_e; gppe_6[:] = gppe

            r_e, K_e = element_res_jac(
                u_e_6, gp_e_6, gppe_6, S_in[e],
                y_e, l_dis, L_en, tau_glob, dt)

            # Scatter back from interleaved to block layout
            for a in range(3):
                R_u[cn_u[a]]   += r_e[2*a]
                R_gp[cn_gp[a]] += r_e[2*a+1]
                for b in range(3):
                    K_uu [cn_u[a],  cn_u[b]  ] += K_e[2*a,   2*b  ]
                    K_ugp[cn_u[a],  cn_gp[b] ] += K_e[2*a,   2*b+1]
                    K_gpu[cn_gp[a], cn_u[b]  ] += K_e[2*a+1, 2*b  ]
                    K_gp [cn_gp[a], cn_gp[b] ] += K_e[2*a+1, 2*b+1]

        R_sys = np.concatenate([R_u, R_gp])
        K_sys = np.block([[K_uu, K_ugp], [K_gpu, K_gp]])
        return R_sys, K_sys

    # ── Helper: GP state update ────────────────────────────────────────────────
    def update_gp_state_fnx(gp_a, gp_prev_a, S_in, dt):
        S_new = S_in.copy()
        for e in range(N_ELEM):
            cn_gp = conn_gp[e]; y_e = x_u_sorted[conn_u[e]]
            jac_e = (y_e[-1] - y_e[0]) / 2.
            for igp, xi in enumerate(XI_GP):
                N, dN_xi = shape_functions(xi); dN = dN_xi / jac_e
                gp_dot   = float((N  @ gp_a[cn_gp] - N  @ gp_prev_a[cn_gp]) / dt)
                gp_dot_y = float((dN @ gp_a[cn_gp] - dN @ gp_prev_a[cn_gp]) / dt)
                dp = float(_dp_jax(jnp.float64(gp_dot), jnp.float64(gp_dot_y),
                                    jnp.float64(l_dis)))
                S_new[e, igp] = float(update_S_jax(
                    jnp.float64(S_in[e, igp]), jnp.float64(dp),
                    jnp.float64(dt), jnp.float64(H_hard)))
        return S_new

    # ── NR step ───────────────────────────────────────────────────────────────
    def solve_step_fnx(gp_prev_a, Gamma_imposed, dt):
        nonlocal u_arr, gp_arr, S_all
        u_w  = u_arr.copy()
        gp_w = gp_arr.copy()

        # Block DOF indices: u part [0..n_u-1], gp part [n_u..n_u+n_gp-1]
        fixed_u   = list(dofs_u_L)  + list(dofs_u_R)
        fixed_gp  = [n_u + d for d in list(dofs_gp_L) + list(dofs_gp_R)]
        fixed_all = fixed_u + fixed_gp

        for _ in range(NR_MAX_ITER):
            Gp  = compute_gamma_p_fnx(gp_w)
            tau = MU * (Gamma_imposed - Gp)

            R_sys, K_sys = assemble_fnx(u_w, gp_w, gp_prev_a, S_all, tau, dt)

            # Apply Dirichlet BCs
            for fd in fixed_all:
                K_sys[fd, :] = 0.; K_sys[:, fd] = 0.
                K_sys[fd, fd] = 1.; R_sys[fd]   = 0.

            if np.linalg.norm(R_sys) < NR_TOL:
                break

            delta = np.linalg.solve(K_sys, -R_sys)
            u_w  += delta[:n_u]
            gp_w += delta[n_u:]

            # Re-impose Dirichlet
            for d in dofs_u_L:   u_w[d]  = 0.
            for d in dofs_u_R:   u_w[d]  = Gamma_imposed * H_STRIP
            for d in dofs_gp_L:  gp_w[d] = gp_prev_a[d]
            for d in dofs_gp_R:  gp_w[d] = gp_prev_a[d]

        u_arr  = u_w
        gp_arr = gp_w
        S_all  = update_gp_state_fnx(gp_w, gp_prev_a, S_all, dt)
        return tau

    # ── Time integration ──────────────────────────────────────────────────────
    Gamma_hist = [0.]; tau_hist = [0.]; gp_profile = None
    dt_fwd = T_END_FWD / n_steps_fwd
    dt_rev = T_END_FWD / n_steps_rev

    for step in range(1, n_steps_fwd + 1):
        Gamma    = GAMMA_MAX * step / n_steps_fwd
        gp_prev_snap = gp_arr.copy()
        tau      = solve_step_fnx(gp_prev_snap, Gamma, dt_fwd)
        Gamma_hist.append(Gamma); tau_hist.append(tau)
        if step == n_steps_fwd:
            gp_profile = (x_gp_sorted.copy(), gp_arr.copy())

    if reverse:
        for step in range(1, n_steps_rev + 1):
            Gamma    = GAMMA_MAX - 2.*GAMMA_MAX * step / n_steps_rev
            gp_prev_snap = gp_arr.copy()
            tau      = solve_step_fnx(gp_prev_snap, Gamma, dt_rev)
            Gamma_hist.append(Gamma); tau_hist.append(tau)

    return np.array(Gamma_hist), np.array(tau_hist), gp_profile

In [35]:
# DISPATCHER
# ═════════════════════════════════════════════════════════════════════════════

def run_simulation(l_dis, L_en, H_hard,
                   n_steps_fwd=N_STEPS_FWD, n_steps_rev=N_STEPS_REV,
                   reverse=True):

    return run_simulation_fenicsx(l_dis, L_en, H_hard,
                                      n_steps_fwd, n_steps_rev, reverse)
    return run_simulation_np(l_dis, L_en, H_hard,
                             n_steps_fwd, n_steps_rev, reverse)

In [36]:
COLORS = ['#1f3a5f', '#e05c2a', '#2d8a4e', '#9b2335', '#5a3e8a', '#c49a00']


def plot_case(ax_ss, ax_prof, results):
    for i, (label, (G_h, tau_h, gp_pr)) in enumerate(results):
        c = COLORS[i % len(COLORS)]
        ax_ss.plot(G_h, tau_h, color=c, label=label)
        if gp_pr is not None:
            nodes_pr, gp_vals = gp_pr
            ax_prof.plot(gp_vals, nodes_pr / H_STRIP, color=c, label=label)
    ax_ss.axhline(0, color='k', lw=0.5, ls='--')
    ax_ss.set_xlabel('Strain Γ'); ax_ss.set_ylabel('Stress τ [MPa]')
    ax_ss.legend(loc='upper left', framealpha=0.7)
    ax_ss.grid(True, lw=0.3, alpha=0.5)
    ax_prof.set_xlabel('Plastic Strain γᵖ'); ax_prof.set_ylabel('y/h')
    ax_prof.legend(loc='center right', framealpha=0.7)
    ax_prof.grid(True, lw=0.3, alpha=0.5)
    ax_prof.set_ylim(0, 1)

In [37]:
backend = "FEniCSx (DOLFINx)" if True else "NumPy"
print("=" * 65)
print("  1D Gradient Plasticity FEM")
print("  Anand, Gurtin, Lele & Gething (JMPS, 2005)")
print(f"  Assembly backend : {backend}")
print(f"  Constitutive     : JAX JIT + jax.jacobian (autodiff tangent)")
print("=" * 65)

cases = {
    "Case 1: Energetic (l=0, H=0)": [
        ("L/h=0.0", 0., 0.0*H_STRIP, 0),
        ("L/h=0.3", 0., 0.3*H_STRIP, 0),
        ("L/h=1.0", 0., 1.0*H_STRIP, 0),
    ],
    "Case 2: Energetic + H=500": [
        ("L/h=0.0", 0., 0.0*H_STRIP, 500),
        ("L/h=0.1", 0., 0.1*H_STRIP, 500),
        ("L/h=0.3", 0., 0.3*H_STRIP, 500),
        ("L/h=1.0", 0., 1.0*H_STRIP, 500),
    ],
    "Case 3: Dissipative (L=0, H=0)": [
        ("l/h=0.0", 0.0*H_STRIP, 0., 0),
        ("l/h=0.1", 0.1*H_STRIP, 0., 0),
        ("l/h=0.5", 0.5*H_STRIP, 0., 0),
        ("l/h=1.0", 1.0*H_STRIP, 0., 0),
    ],
    "Case 4: Dissipative + H=500": [
        ("l/h=0.0", 0.0*H_STRIP, 0., 500),
        ("l/h=0.1", 0.1*H_STRIP, 0., 500),
        ("l/h=0.5", 0.5*H_STRIP, 0., 500),
        ("l/h=1.0", 1.0*H_STRIP, 0., 500),
    ],
    "Case 5: Combined": [
        ("no gradient",              0.,          0.,          0),
        ("l/h=0.5",                  0.5*H_STRIP, 0.,          0),
        ("l/h=0.5, L/h=1.0",         0.5*H_STRIP, 1.0*H_STRIP, 0),
        ("l/h=0.5, L/h=1.0, H=200",  0.5*H_STRIP, 1.0*H_STRIP, 200),
    ],
}

all_results = {}
for case_name, params in cases.items():
    print(f"\n{case_name}")
    res_list = []
    for label, l, L, H in params:
        print(f"  {label} ...", end="", flush=True)
        res = run_simulation(l, L, H)
        res_list.append((label, res))
        print(" done")
    all_results[case_name] = res_list

# ── Figure ────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family': 'serif', 'font.size': 9,
                        'axes.titlesize': 10, 'axes.labelsize': 9,
                        'legend.fontsize': 7.5, 'lines.linewidth': 1.5})

fig = plt.figure(figsize=(16, 18))
fig.patch.set_facecolor('#fafaf8')
gs  = gridspec.GridSpec(5, 2, figure=fig, hspace=0.45, wspace=0.32,
                        left=0.07, right=0.96, top=0.94, bottom=0.04)

for row, (case_name, res_list) in enumerate(all_results.items()):
    ax_ss   = fig.add_subplot(gs[row, 0])
    ax_prof = fig.add_subplot(gs[row, 1])
    ax_ss.set_title(f"{case_name} — stress–strain",   fontweight='bold')
    ax_prof.set_title(f"{case_name} — γᵖ profile at Γ=0.1", fontweight='bold')
    plot_case(ax_ss, ax_prof, res_list)

for ax in fig.axes:
    ax.set_facecolor('#fafaf8')

fig.suptitle(
    "1D Gradient Plasticity FEM  —  Anand, Gurtin, Lele & Gething (JMPS, 2005)\n"
    f"Assembly: {backend}  |  Constitutive: JAX JIT + jax.jacobian  |  "
    "CG2 elements  |  Backward-Euler",
    fontsize=10, fontweight='bold', y=0.972)

out = '/mnt/user-data/outputs/gradient_plasticity_results.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#fafaf8')
plt.close()
print(f"\nFigure saved → {out}")
print("Done.")

  1D Gradient Plasticity FEM
  Anand, Gurtin, Lele & Gething (JMPS, 2005)
  Assembly backend : FEniCSx (DOLFINx)
  Constitutive     : JAX JIT + jax.jacobian (autodiff tangent)

Case 1: Energetic (l=0, H=0)
  L/h=0.0 ...

KeyboardInterrupt: 